# CNN (Convolutional Neural Networks)

이 노트북에서는 합성곱 신경망(CNN)의 원리를 이해하고 이미지 분류 모델을 구현합니다.

## 학습 목표
1. 합성곱 연산의 이해
2. CNN 아키텍처 구현 (LeNet, AlexNet, ResNet)
3. CIFAR-10 이미지 분류
4. 데이터 증강 기법
5. Transfer Learning

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 합성곱 연산 이해하기

합성곱(Convolution)은 이미지의 특징을 추출하는 핵심 연산입니다.

In [ ]:
# 합성곱 연산 시각화
def visualize_convolution():
    # 입력 이미지 (간단한 엣지)
    input_image = torch.zeros(1, 1, 7, 7)
    input_image[0, 0, :, 3] = 1  # 수직선
    
    # 엣지 검출 커널
    edge_kernel = torch.tensor([[-1, 0, 1],
                                [-2, 0, 2],
                                [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    
    # 합성곱 적용
    conv = nn.Conv2d(1, 1, 3, bias=False)
    conv.weight.data = edge_kernel
    output = conv(input_image)
    
    # 시각화
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    axes[0].imshow(input_image.squeeze(), cmap='gray')
    axes[0].set_title('Input Image')
    axes[0].axis('off')
    
    axes[1].imshow(edge_kernel.squeeze(), cmap='RdBu')
    axes[1].set_title('Sobel Kernel')
    axes[1].axis('off')
    
    axes[2].imshow(output.detach().squeeze(), cmap='gray')
    axes[2].set_title('Output (Edge Detected)')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_convolution()

In [ ]:
# Conv2d 파라미터 이해
print("Conv2d 예제:")
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
print(f"입력 채널: 3 (RGB)")
print(f"출력 채널: 16 (특징 맵 수)")
print(f"커널 크기: 3x3")
print(f"스트라이드: 1")
print(f"패딩: 1")

# 가중치 모양
print(f"\n가중치 shape: {conv.weight.shape}")
print(f"파라미터 수: {conv.weight.numel() + conv.bias.numel()}")

# 출력 크기 계산
x = torch.randn(1, 3, 32, 32)
y = conv(x)
print(f"\n입력 shape: {x.shape}")
print(f"출력 shape: {y.shape}")

## 2. CIFAR-10 데이터셋 로드

In [ ]:
# 데이터 증강 및 정규화
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# 데이터셋 다운로드
train_dataset = torchvision.datasets.CIFAR10(
    root='../data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(
    root='../data', train=False, download=True, transform=transform_test)

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

# 클래스 이름
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f"학습 샘플: {len(train_dataset)}")
print(f"테스트 샘플: {len(test_dataset)}")

In [ ]:
# 데이터 시각화
def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

# 샘플 이미지
dataiter = iter(train_loader)
images, labels = next(dataiter)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = images[i] / 2 + 0.5
    ax.imshow(np.transpose(img.numpy(), (1, 2, 0)))
    ax.set_title(classes[labels[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 3. CNN 모델 구현

In [ ]:
class SimpleCNN(nn.Module):
    """간단한 CNN 모델"""
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        # 특징 추출
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        
        # 분류기
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # Conv Block 1: 32x32 -> 16x16
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        # Conv Block 2: 16x16 -> 8x8
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        # Conv Block 3: 8x8 -> 4x4
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # FC layers
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
print(model)
print(f"\n파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class ResidualBlock(nn.Module):
    """ResNet 잔차 블록"""
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)  # Skip connection
        out = F.relu(out)
        return out

class ResNet(nn.Module):
    """ResNet for CIFAR-10"""
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()
        self.in_channels = 64
        
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)
        
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)
    
    def _make_layer(self, out_channels, num_blocks, stride):
        layers = [ResidualBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avg_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

resnet = ResNet().to(device)
print(f"ResNet 파라미터 수: {sum(p.numel() for p in resnet.parameters()):,}")

## 4. 모델 학습

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    for epoch in range(epochs):
        # Training
        model.train()
        running_loss, correct, total = 0, 0, 0
        
        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}'):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        
        # Testing
        model.eval()
        running_loss, correct, total = 0, 0, 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        
        test_loss = running_loss / len(test_loader)
        test_acc = 100. * correct / total
        test_losses.append(test_loss)
        test_accs.append(test_acc)
        
        scheduler.step()
        
        print(f'Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, '
              f'Test Loss={test_loss:.4f}, Test Acc={test_acc:.2f}%')
    
    return train_losses, train_accs, test_losses, test_accs

In [ ]:
# 모델 학습
model = SimpleCNN().to(device)
train_losses, train_accs, test_losses, test_accs = train_model(
    model, train_loader, test_loader, epochs=10, lr=0.001
)

In [ ]:
# 학습 결과 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss')
ax1.plot(test_losses, label='Test Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve')
ax1.legend()
ax1.grid(True)

ax2.plot(train_accs, label='Train Acc')
ax2.plot(test_accs, label='Test Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curve')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 5. 특징 맵 시각화

In [ ]:
# 중간 특징 맵 시각화
def visualize_feature_maps(model, image):
    model.eval()
    activations = []
    
    def hook_fn(module, input, output):
        activations.append(output.detach().cpu())
    
    # 첫 번째 conv layer에 hook 등록
    hook = model.conv1.register_forward_hook(hook_fn)
    
    with torch.no_grad():
        _ = model(image.unsqueeze(0).to(device))
    
    hook.remove()
    
    # 특징 맵 시각화
    feature_maps = activations[0].squeeze()
    
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        if i < feature_maps.size(0):
            ax.imshow(feature_maps[i], cmap='viridis')
        ax.axis('off')
    
    plt.suptitle('Feature Maps from First Conv Layer')
    plt.tight_layout()
    plt.show()

# 테스트 이미지로 시각화
test_image = test_dataset[0][0]
visualize_feature_maps(model, test_image)

## 연습 문제

1. **더 깊은 네트워크**: Conv layer를 추가하고 성능 변화를 관찰하세요.
2. **다른 정규화**: Dropout 비율을 변경하거나 L2 정규화를 추가해보세요.
3. **데이터 증강**: 다른 증강 기법 (회전, 색상 변환 등)을 시도해보세요.
4. **ResNet 학습**: ResNet 모델을 학습하고 SimpleCNN과 비교해보세요.

## 다음 단계

- [03_rnn_lstm.ipynb](./03_rnn_lstm.ipynb): 순환 신경망으로 시퀀스 데이터 처리
- [04_transformer.ipynb](./04_transformer.ipynb): Transformer 아키텍처 구현